# Getting Started with uniqx

**uniqx** is a hardware-agnostic computation platform. You write Python code that
describes operations (arithmetic, linear algebra, eigensolvers, time evolution, etc.),
and the SDK **traces** those operations into an intermediate representation (IR).
This module is then sent to **uniqx**, which compiles and routes the computation
to the best available hardware -- CPU, GPU, or QPU -- entirely server-side.

Your Python code never runs the math itself. It only *describes* the computation.
uniqx handles compilation, optimization, and execution.

This notebook walks through the core concepts and API.

## 1. Installation and Imports

The uniqx SDK exposes operations through `uniqx.ops`, tracing through
`uniqx.tracing`, and type helpers through `uniqx.types`. Execution helpers
(`connect`, `submit`, `get`, `preflight`, `parse_result`, etc.) are
re-exported from the top-level `uniqx` package.

In [ ]:
# Copyright (c) 2024-2026 ORIQX AG/LTD/SAS. All rights reserved.

from uniqx import ops, tracing, types
from uniqx import (
    connect,
    submit,
    get,
    preflight,
    parse_result,
    parse_buffer_view,
    fmt_mat,
    fmt_vec,
    fmt_scalar,
)

print("uniqx SDK imported successfully")

## 2. Core Concepts

The uniqx workflow has four stages:

1. **Trace** -- Decorate a Python function with `@tracing.to_module`. When called,
 the function is *traced* (not executed). The SDK records every operation and
 builds an IR module.
2. **Preflight** -- Send the module to uniqx for analysis. uniqx
 evaluates hardware options (CPU, GPU, QPU) and returns Pareto-optimal
 execution plans with estimated cost, time, error, and carbon.
3. **Submit** -- Submit the module for execution on the selected hardware.
 Returns a job ID.
4. **Get** -- Poll/wait for results. Parse the binary payload back into numbers.

Arguments passed to a traced function become *symbolic parameters*. uniqx
can accept concrete values at submit time via `runtime_inputs`.

## 3. Your First Traced Module

Let's start with simple scalar arithmetic. The `@tracing.to_module` decorator
turns a function into a module-producing function. When you call it, you get
back an `ir.Module` object -- not a number.

In [ ]:
@tracing.to_module(name="arithmetic_demo")
def arithmetic_demo(a, b):
    """Demonstrate basic arithmetic ops."""
    sum_ab = a + b           # ops.add
    diff_ab = a - b          # ops.sub
    prod_ab = a * b          # ops.mul
    quot_ab = a / b          # ops.div
    neg_a = ops.neg(a)       # negation
    abs_neg = ops.abs(neg_a) # absolute value
    power = ops.pow(a, b)    # exponentiation
    return sum_ab, diff_ab, prod_ab, quot_ab, neg_a, abs_neg, power

# Trace with example arguments (shape/type are inferred, values are symbolic)
module = arithmetic_demo(3.0, 2.0)

# Inspect the IR text
print("Module name:", module.name)
print("Number of functions:", len(module.functions))
print("Number of ops:", len(module.functions[0].ops))
print()
print("--- IR Text ---")
print(module.to_text())

Notice that calling `arithmetic_demo(3.0, 2.0)` did **not** compute `3.0 + 2.0 = 5.0`.
Instead, it recorded an `add` op with two `f64` parameters. The actual computation
happens on the server when you submit the module.

## 4. Connecting to uniqx

To execute a module, you need access to a uniqx service. Set `UNIQX_GATEWAY` to the endpoint (for example `api.oriqx.com:443` for the hosted service). If unset, the starter notebooks default to `api.oriqx.com:443` (the hosted hackathon gateway). Use `localhost:50050` only for local dev.

In [ ]:
import os
from uniqx import connect, login

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)

## 5. Preflight -- Analyzing Execution Options

`preflight()` sends the module to uniqx for analysis *without* executing it.
uniqx returns a list of Pareto-optimal execution plans. Each option includes
estimated time, cost, error rate, carbon footprint, and per-node hardware assignments.

The result is a `PreflightResult` -- a list you can iterate over, with a `.job_id`
attribute and helper methods like `.summary()` and `.recommended`.

In [ ]:
module = arithmetic_demo(3.0, 2.0)
options = preflight(module, client=client)

print("Preflight job ID:", options.job_id)
print("Number of execution options:", len(options))
print()
print(options.summary())
print()

# The recommended option
rec = options.recommended
if rec:
    print("Recommended:", rec["label"])

## 6. Submit and Get -- Running the Computation

`submit()` sends the module for execution and returns a job ID.
`get()` waits for the job to complete and returns the result payload.

You can optionally pass `runtime_inputs` to provide concrete values for the
module parameters. Use `fmt_scalar`, `fmt_vec`, or `fmt_mat` to encode values
in uniqx's buffer-view format.

In [ ]:
# Submit with concrete runtime inputs
module = arithmetic_demo(3.0, 2.0)
job_id = submit(
    module,
    client=client,
    runtime_inputs=[3.0, 2.0],
)
print("Job ID:", job_id)

# Wait for result
result = get(job_id, client=client)
print("Job state:", result.get("state"))

# Parse the binary payload into named outputs
payload = result.get("payload") or result.get("result_payload")
if payload:
    names = ["sum", "diff", "prod", "quot", "neg", "abs", "pow"]
    parsed = parse_result(payload, names)
    for name, (dims, dtype, values) in parsed.items():
        print(f"  {name}: {values}")

## 7. Arithmetic Operations

The SDK provides element-wise arithmetic ops that work on scalars and tensors.
You can use Python operators (`+`, `-`, `*`, `/`) on traced values, or call
the explicit functions from `ops`.

| Op | Python operator | Explicit function |
|----|-----------------|-------------------|
| Add | `a + b` | `ops.add(a, b)` |
| Subtract | `a - b` | `ops.sub(a, b)` |
| Multiply | `a * b` | `ops.mul(a, b)` |
| Divide | `a / b` | `ops.div(a, b)` |
| Negate | `-a` | `ops.neg(a)` |
| Absolute | -- | `ops.abs(a)` |
| Power | -- | `ops.pow(a, b)` |
| Remainder | -- | `ops.rem(a, b)` |
| Min/Max | -- | `ops.min(a, b)` / `ops.max(a, b)` |

In [ ]:
@tracing.to_module(name="tensor_arithmetic")
def tensor_arithmetic(A, B):
    """Element-wise arithmetic on 2x2 matrices."""
    return A + B, A * B, ops.pow(A, 2.0)

# Trace with 2x2 matrix arguments
mat_a = [[1.0, 2.0], [3.0, 4.0]]
mat_b = [[5.0, 6.0], [7.0, 8.0]]
module = tensor_arithmetic(mat_a, mat_b)

print("Traced tensor arithmetic module:")
print(f"  Parameters: {len(module.functions[0].params)}")
print(f"  Ops: {len(module.functions[0].ops)}")
print(f"  Results: {len(module.functions[0].results)}")

## 8. Linear Algebra Operations

The `ops` module includes standard linear algebra operations. These are traced
as IR nodes and compiled/executed server-side.

- `ops.matmul(A, B)` -- matrix multiplication
- `ops.dot(a, b)` -- inner product / batched dot product
- `ops.kron(A, B)` -- Kronecker product (tensor product)
- `ops.transpose(A, permutation)` -- axis permutation
- `ops.diag(v, result_type=...)` -- diagonal matrix from vector
- `ops.eye(n, result_type=...)` -- identity matrix

In [ ]:
@tracing.to_module(name="linalg_demo")
def linalg_demo(A, B, v):
    """Demonstrate linear algebra operations."""
    # Matrix multiply: [2,2] x [2,2] -> [2,2]
    product = ops.matmul(A, B)

    # Matrix-vector multiply: [2,2] x [2] -> [2]
    mv = ops.matmul(A, v)

    # Dot product of two vectors: [2] . [2] -> scalar
    dp = ops.dot(v, v)

    # Kronecker product: [2,2] x [2,2] -> [4,4]
    kr = ops.kron(A, B)

    # Transpose: swap axes 0 and 1
    At = ops.transpose(A, permutation=[1, 0])

    return product, mv, dp, kr, At

module = linalg_demo(
    [[1.0, 2.0], [3.0, 4.0]],  # A: 2x2
    [[5.0, 6.0], [7.0, 8.0]],  # B: 2x2
    [1.0, 0.0],                  # v: 2-vector
)

print("Linear algebra module traced successfully")
print(f"  Ops: {len(module.functions[0].ops)}")
for r in module.functions[0].results:
    print(f"  Result {r.id}: {r.type.to_dict()}")

## 9. Primitives: eigs -- Eigenvalue Decomposition

`ops.eigs(A, k=..., which=..., hermitian=...)` computes the leading
eigenpairs of a matrix. It returns `(eigenvalues, eigenvectors)`.

uniqx routes this to the best available solver:
- **CPU**: dense LAPACK or iterative Lanczos
- **GPU**: cuSOLVER
- **QPU**: VQE or quantum phase estimation

Let's compute eigenvalues of a 4x4 symmetric (Hermitian) matrix.

In [ ]:
@tracing.to_module(name="eigs_demo")
def eigs_demo(H):
    """Compute the 2 smallest eigenvalues of a 4x4 Hermitian matrix."""
    eigenvalues, eigenvectors = ops.eigs(
        H,
        k=2,
        which="smallest",
        hermitian=True,
    )
    return eigenvalues, eigenvectors

# A 4x4 symmetric matrix (real Hermitian)
H = [
    [ 2.0, -1.0,  0.0,  0.0],
    [-1.0,  2.0, -1.0,  0.0],
    [ 0.0, -1.0,  2.0, -1.0],
    [ 0.0,  0.0, -1.0,  2.0],
]

module = eigs_demo(H)

print("Eigenvalue module traced")
print("IR text (excerpt):")
text = module.to_text()
for line in text.splitlines()[:20]:
    print(" ", line)

## 10. Primitives: expv -- Matrix Exponential

`ops.expv(A, v, t)` computes `exp(t * A) @ v` -- the action of a matrix
exponential on a vector. This is the core primitive for time evolution in
quantum mechanics and differential equations.

uniqx can lower this to:
- **Classical**: Krylov/Lanczos subspace methods or dense expm + matmul
- **Quantum**: time-evolution circuit synthesis

Key parameters:
- `hermitian=True` enables symmetric Lanczos (faster, more stable)
- `precision` sets the target numerical accuracy

In [ ]:
@tracing.to_module(name="expv_demo")
def expv_demo(H, psi0, t):
    """Time-evolve a state: psi(t) = exp(t * H) * psi0."""
    psi_t = ops.expv(
        H, psi0, t,
        hermitian=True,
        precision=1e-8,
    )
    return psi_t

# 4x4 Hamiltonian and initial state
H = [
    [ 0.0, -1.0,  0.0,  0.0],
    [-1.0,  0.0, -1.0,  0.0],
    [ 0.0, -1.0,  0.0, -1.0],
    [ 0.0,  0.0, -1.0,  0.0],
]
psi0 = [1.0, 0.0, 0.0, 0.0]  # initial state: site 0
t = 1.0                        # evolution time

module = expv_demo(H, psi0, t)
print("expv module traced")
print(f"  Ops: {len(module.functions[0].ops)}")

## 11. Primitives: expect -- Expectation Values

`ops.expect(op, state)` computes the expectation value `<state| op |state>`.
This is the standard quantum-mechanical observable measurement.

- With `shots=None` (default): exact computation via linear algebra
- With `shots=N`: stochastic estimate via sampling (useful for QPU backends)

In [ ]:
@tracing.to_module(name="expect_demo")
def expect_demo(H, psi):
    """Compute <psi|H|psi> -- the energy expectation value."""
    energy = ops.expect(H, psi)
    return energy

# Hamiltonian and state
H = [
    [ 1.0, -0.5,  0.0,  0.0],
    [-0.5,  1.0, -0.5,  0.0],
    [ 0.0, -0.5,  1.0, -0.5],
    [ 0.0,  0.0, -0.5,  1.0],
]
psi = [0.5, 0.5, 0.5, 0.5]  # uniform superposition

module = expect_demo(H, psi)
print("Expectation value module traced")
print(f"  Output type: {module.functions[0].results[0].type.to_dict()}")

## 12. Control Flow: fori_loop

The SDK supports structured control flow that is traced into the IR graph.
Unlike Python loops (which would unroll), these emit a single loop node that
uniqx compiles into an efficient loop on the target hardware.

- `ops.fori_loop(lower, upper, body_fn, init_val)` -- JAX-style counted loop
- `ops.cond(pred, true_fn, false_fn, *operands)` -- conditional branch

The `body_fn(i, carry)` receives the loop index and a carry value, and must
return the updated carry.

In [ ]:
@tracing.to_module(name="loop_demo")
def loop_demo(x):
    """Sum x + x + x + ... (10 times) using a traced loop."""
    def body(i, carry):
        return carry + x

    result = ops.fori_loop(0, 10, body, 0.0)
    return result

module = loop_demo(3.14)
print("Loop module traced")
print(f"  Functions: {len(module.functions)} (main + body)")
for fn in module.functions:
    print(f"    {fn.name}: {len(fn.ops)} ops, {len(fn.params)} params")

## 13. Hardware Routing and Preflight Analysis

uniqx performs multi-objective optimization to find the best hardware
assignment for each operation in your computation graph. The preflight result
shows the Pareto frontier of execution options.

Each option includes:
- `label` -- strategy name (e.g., `cpu-only`, `cpu+gpu`, `cpu+qpu`)
- `total_time` -- estimated compute time
- `total_cost_usd` -- estimated cost in USD
- `max_error_rate` -- worst-case numerical error
- `total_carbon_g` -- CO2 footprint in grams
- `node_assignments` -- per-node hardware mapping

You can select a specific option and pass it back to `submit()` using
`preflight_job_id` and `option_idx` to skip re-scoring.

In [ ]:
# Preflight the eigenvalue module to see routing options
H = [
    [ 2.0, -1.0,  0.0,  0.0],
    [-1.0,  2.0, -1.0,  0.0],
    [ 0.0, -1.0,  2.0, -1.0],
    [ 0.0,  0.0, -1.0,  2.0],
]
module = eigs_demo(H)

options = preflight(module, client=client)
print(options.summary())
print()
print("Needs GPU?", options.needs_gpu)
print("Needs QPU?", options.needs_qpu)

# Select a specific option by label
rec = options.recommended
if rec:
    print(f"\nRecommended option: {rec['label']}")
    print(f"  Node assignments: {rec.get('node_assignments', {})}")

## 14. Encoding Runtime Inputs

When submitting a module, you can provide concrete values for the parameters.
The SDK auto-encodes Python values, but you can also use the low-level
buffer-view format directly:

- `fmt_scalar(value, dtype)` -- encode a scalar, e.g. `"f64= 3.14"`
- `fmt_vec(values, n, dtype)` -- encode a vector, e.g. `"4xf64= 1 0 0 0"`
- `fmt_mat(values, rows, cols, dtype)` -- encode a matrix, e.g. `"2x2xf64= 1 0 0 1"`

The inverse is `parse_buffer_view(line)` which returns `(dims, dtype, values)`.

In [ ]:
# Encoding examples
print("Scalar:", fmt_scalar(3.14))
print("Vector:", fmt_vec([1.0, 2.0, 3.0], 3))
print("Matrix:", fmt_mat([[1.0, 0.0], [0.0, 1.0]], 2, 2))
print()

# Parsing a buffer-view line
parsed = parse_buffer_view("2x2xf64= 1.0 0.0 0.0 1.0")
if parsed:
    dims, dtype, values = parsed
    print(f"Parsed: dims={dims}, dtype={dtype}, values={values}")

## 15. Transcendental Operations

The SDK includes standard transcendental functions that work element-wise
on scalars and tensors:

`ops.sin`, `ops.cos`, `ops.tan`, `ops.exp`, `ops.log`, `ops.tanh`,
`ops.sqrt`, `ops.rsqrt`, `ops.erf`, `ops.logistic`

In [ ]:
@tracing.to_module(name="transcendental_demo")
def transcendental_demo(x):
    """Compose transcendental operations."""
    a = ops.exp(x)
    b = ops.log(a)        # log(exp(x)) = x
    c = ops.sqrt(x * x)   # sqrt(x^2) = |x|
    d = ops.sin(x) * ops.sin(x) + ops.cos(x) * ops.cos(x)  # sin^2 + cos^2 = 1
    return a, b, c, d

module = transcendental_demo(1.5)
print("Transcendental module traced")
print(f"  Ops: {len(module.functions[0].ops)}")

## 16. Operator Embedding and Kronecker Products

For multi-qubit/multi-site systems, the SDK provides:

- `ops.kron(A, B)` -- Kronecker product
- `ops.embed(op, qubit, n_qubits)` -- embed single-qubit op into n-qubit space
- `ops.two_body(opA, opB, qa, qb, n_qubits)` -- embed two-body interaction

Pre-defined Pauli matrices are available as `ops.SX`, `ops.SY`, `ops.SZ`, `ops.I2`.

In [ ]:
@tracing.to_module(name="hamiltonian_construction")
def hamiltonian_construction(J):
    """Build a 2-site Heisenberg Hamiltonian: H = J * (XX + YY + ZZ)."""
    n_qubits = 2

    # Two-body terms: S_i . S_j for i=0, j=1
    xx = ops.two_body(ops.SX, ops.SX, 0, 1, n_qubits)
    yy = ops.two_body(ops.SY, ops.SY, 0, 1, n_qubits)
    zz = ops.two_body(ops.SZ, ops.SZ, 0, 1, n_qubits)

    # H = J * (XX + YY + ZZ)
    H = J * (xx + yy + zz)
    return H

module = hamiltonian_construction(1.0)
print("Hamiltonian construction module traced")
print(f"  Ops: {len(module.functions[0].ops)}")

## 17. Complete Example: Eigenvalues of a Spin Chain

Let's put it all together. We will:
1. Build a 2-site Heisenberg Hamiltonian
2. Compute its lowest eigenvalue (ground state energy)
3. Compute the expectation value of the ZZ operator
4. Submit everything to uniqx and parse the results

This demonstrates how multiple primitives compose within a single traced module.

In [ ]:
@tracing.to_module(name="spin_chain_analysis")
def spin_chain_analysis(J):
    """Full spin chain analysis: eigenvalues + expectation value."""
    n_qubits = 2

    # Build Hamiltonian
    xx = ops.two_body(ops.SX, ops.SX, 0, 1, n_qubits)
    yy = ops.two_body(ops.SY, ops.SY, 0, 1, n_qubits)
    zz = ops.two_body(ops.SZ, ops.SZ, 0, 1, n_qubits)
    H = J * (xx + yy + zz)

    # Compute 2 smallest eigenvalues
    eigenvalues, eigenvectors = ops.eigs(H, k=2, which="smallest", hermitian=True)

    # Measure <ZZ> on the ground state (first eigenvector)
    zz_op = ops.two_body(ops.SZ, ops.SZ, 0, 1, n_qubits)

    return eigenvalues, eigenvectors

module = spin_chain_analysis(1.0)
print("Spin chain module traced successfully")
print(f"  Functions: {len(module.functions)}")
print(f"  Total ops: {len(module.functions[0].ops)}")
print(f"  Results: {len(module.functions[0].results)}")

In [ ]:
# Submit and execute the spin chain analysis
job_id = submit(
    module,
    client=client,
    runtime_inputs=[1.0],  # J = 1.0
)
print("Submitted job:", job_id)

result = get(job_id, client=client)
print("Job state:", result.get("state"))

payload = result.get("payload") or result.get("result_payload")
if payload:
    parsed = parse_result(payload, ["eigenvalues", "eigenvectors"])
    if "eigenvalues" in parsed:
        dims, dtype, vals = parsed["eigenvalues"]
        print(f"Eigenvalues: {vals}")
    if "eigenvectors" in parsed:
        dims, dtype, vals = parsed["eigenvectors"]
        print(f"Eigenvectors shape: {dims}")

## 18. Preflight with Option Selection

For production workflows, you typically preflight first, inspect the options,
then submit with the chosen option. The `preflight_job_id` and `option_idx`
parameters let uniqx reuse the cached analysis, avoiding re-scoring.

In [ ]:
# Preflight -> select option -> submit with cached analysis
module = spin_chain_analysis(1.0)

options = preflight(module, client=client)
print("Preflight options:")
print(options.summary())

# Pick the recommended option
chosen = options.recommended
if chosen:
    print(f"\nSubmitting with option: {chosen['label']} (idx={chosen['_idx']})")
    job_id = submit(
        module,
        client=client,
        runtime_inputs=[1.0],
        preflight_job_id=options.job_id,
        option_idx=chosen["_idx"],
    )
    result = get(job_id, client=client)
    print("Job state:", result.get("state"))

## 19. Type System

The `types` module provides helpers for constructing IR types used in ops
that require explicit type annotations (e.g., `result_type` parameters).

- `types.scalar(dtype)` -- scalar type, e.g. `types.scalar("f64")`
- `types.tensor(dtype, shape)` -- tensor type, e.g. `types.tensor("f64", [4, 4])`
- `types.quantum(qubits)` -- quantum register type

Supported dtypes: `f32`, `f64`, `i32`, `i64`, `bool`

In [ ]:
# Type construction examples
s = types.scalar("f64")
print("Scalar f64:", s.to_dict())

t = types.tensor("f64", [3, 3])
print("Tensor 3x3 f64:", t.to_dict())

v = types.tensor("f64", [4])
print("Vector 4 f64:", v.to_dict())

q = types.quantum(2)
print("2-qubit register:", q.to_dict())

## 20. Summary

You have learned the core uniqx workflow:

1. **Import**: `from uniqx import ops, tracing, types, connect, submit, get, preflight, parse_result`
2. **Trace**: Decorate with `@tracing.to_module(name="...")` and call with example arguments
3. **Connect**: `client = connect(os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443"))` (after `uniqx.login(...)` in an earlier cell)
4. **Preflight**: `options = preflight(module, client=client)` to see hardware routing
5. **Submit**: `job_id = submit(module, client=client, runtime_inputs=[...])` to execute
6. **Get**: `result = get(job_id, client=client)` to fetch results
7. **Parse**: `parse_result(payload, names)` to decode the binary payload

Key operations available:
- **Arithmetic**: `add`, `sub`, `mul`, `div`, `neg`, `abs`, `pow`, `rem`
- **Transcendental**: `sin`, `cos`, `exp`, `log`, `sqrt`, `tanh`, `erf`
- **Linear algebra**: `matmul`, `dot`, `kron`, `transpose`, `diag`, `eye`
- **Primitives**: `eigs`, `expv`, `expect`, `time_evolve`, `linear_solve`
- **Control flow**: `fori_loop`, `cond`, `while_loop`, `scan`
- **Shape**: `reshape`, `slice`, `concatenate`

All computation is server-side. Python only traces and submits.